# Гир 1 — симулятор фиксированной стратегии

Ноутбук `model.ipynb` — **симулятор гира 1** в треке модели: прогон по историческому спреду `Bybit`/`OKX` из `parquet`. Это не боевой бот и не живой контур; цель — воспроизводимая проверка правил на истории (`docs/strategy-gears.md`). Параметры не обучаются: один заранее выбранный снимок в `CONFIG`.

Метрика по **закрытым** сделкам: `sum(open_spread + close_spread)` с учётом `position_frac` и `fee_rate`. Сделка со статусом `open` на конце ряда в метрику не входит.

## Логика одного тика

На каждом тике: если есть `pending` / `fill` по `Trade_Lat` — сначала исполнение, новые сигналы на тике запрещены. Иначе проверяются пороги open/close long/short → `Gate A` → `Gate B` → объём → schedule fill; отказ гейта или конец ветки → следующий тик. Закрытие с fill обновляет `metric`. Блок-схема — в следующей ячейке.

Кратко по слоям: порог → `Gate A` (свежесть / задержка доставки) → `Gate B` (скользящее среднее ≥ доля × порог) → гейт объёма (`Check_volume`) → планирование исполнения. Пока висит `pending` или на тике только что был `fill`, вход в новые сигналы заблокирован.

## `VARIATION` и `HYPER`

| Блок | Что внутри | Роль |
|------|------------|------|
| `VARIATION` | пороги `thresh_*`, доли усреднения `open_frac` / `close_frac` | **вариация** гира 1: в гире 2 её будут искать; здесь — зафиксированный вектор |
| `HYPER` | потолки задержки, `avg_window_sec`, `Trade_Lat`, `fee_rate`, `position_frac`, `Check_volume` | **гиперпараметры** профиля симуляции: на время гира 1 заморожены |
| `DATA` | монета, даты, `df`, лимит точек графика | входной ряд; смена периода → пересчёт p95 в `HYPER` |

Единый источник правды: `CONFIG = {variation, hyper, data}`.

### Как задаются гиперпараметры (сжато)

| Параметр | Правило |
|----------|---------|
| `max_latency_okx_ms` / `max_latency_bybit_ms` | **p95** по объединённому `df` периода |
| `max_freshness_ms` | `None` (окно на будущее; сейчас выключено) |
| `avg_window_sec` | вручную; должно быть ≫ характерного прострела (сейчас `2.0`) |
| `Trade_Lat` | временная константа (мс); позже — из замера скорости ордера |
| `position_frac` | `1.0` до гира 2.5 (не предмет поиска) |
| `fee_rate` | среднее за ногу: `(0.0005 + 0.001) / 2 = 0.00075` |
| `Check_volume` | `False` |

Комиссия: `fee_rate` × 2 ноги на открытии × 2 на закрытии → полный круг ≈ **0.3%** номинала **одной** биржи (`4 × 0.00075`), не сумма двух независимых номиналов.

Данные: партиции `event_date` в `[start_date, end_date]` включительно. Пропуски дней — предупреждение. Смена монеты или периода → перезагрузка `df` и пересчёт p95 → `HYPER`.

## Контракт симулятора

**Приближения.** Одна `Trade_Lat` на обе биржи; исполнение на первом тике с `event_local_ts_ms ≥ signal_ts + Trade_Lat` (`0` → тик сигнала). Объём при включённом гейте проверяется на тике **сигнала**, не на тике исполнения. Проскальзывание и частичное исполнение не моделируются: цена = спред на тике `fill`.

**Честный прогон.** Одни и те же правила, замороженный вектор `VARIATION` + `HYPER`, воспроизводимый ряд из `parquet`, учтённые комиссии из профиля исполнения. Итог метрики на короткой истории — проверка кода и согласованности правил, не доказательство торгового преимущества.

**Сознательно не моделируем.** Живую очередь ордеров, сетевые сбои, асинхронный бот, метрики в реальном времени, встраивание в боевой контур. Это зона трека 3, не условие закрытия гира 1.

## Режимы сравнения

Одинаковые числа из `VARIATION` / `HYPER`; отличаются только флаги:

1. **`OFF`** — `gates=False`, `execute=False` (сигналы без гейтов и без задержки исполнения)
2. **`gear1`** — `gates=True`, `execute=True` (полный профиль гира 1; на графике стратегии — `result_gear1`)
3. **`OFF+Trade_Lat`** — `gates=False`, `execute=True` (без гейтов, но с задержкой исполнения)

Инвариант: при выключенных гейтах добавление `Trade_Lat` не увеличивает число закрытых сделок относительно `OFF`.

Порядок ячеек: блок-схема → конфиг → функции → прогон и график → приложение (задержки) → валидация сделок. После смены монеты, периода или порогов — перезапуск с ячейки конфига.


Блок-схема логики симуляции (ячейка ниже)

In [ ]:
# Блок-схема логики бэктест-симуляции (graphviz)
# Фактически по циклу `run_backtest`: pending/fill → elif пороги → Gate A/B → объём → Trade_Lat.
from graphviz import Digraph

dot = Digraph(comment="run_backtest — фактическая схема")
dot.format = "png"
dot.attr(rankdir="TB", fontsize="11", nodesep="0.3", ranksep="0.45")
dot.attr("node", fontname="Helvetica", fontsize="10", shape="box")
dot.attr("edge", fontname="Helvetica", fontsize="9")

# --- Конфиг (не поток тика) ---
with dot.subgraph(name="cluster_cfg") as c:
    c.attr(label="входы: CONFIG", style="dashed", color="#888888", fontsize="10")
    c.node(
        "cfg",
        "пороги / frac · Gate A/B ·\nTrade_Lat · Check_volume · fees",
        shape="note",
        style="filled",
        fillcolor="#f5f5f5",
        fontsize="9",
    )

# --- Цикл по тикам ---
with dot.subgraph(name="cluster_tick") as c:
    c.attr(label="тик / цикл по строкам", style="rounded", color="#4a6fa5", fontsize="11")
    c.node("start", "Старт тика i\n(строка ряда)", style="filled", fillcolor="#d6e4f0")
    c.node("pend_fill", "pending и\ni == fill_i ?", shape="diamond", style="filled", fillcolor="#ffe6a7")
    c.node(
        "exec_now",
        "исполнить pending fill\n(_execute_fill)\nfilled_now = True",
        style="filled",
        fillcolor="#c8e6c9",
    )
    c.node("pend_wait", "pending ещё\nждёт fill?", shape="diamond", style="filled", fillcolor="#ffe6a7")
    c.node(
        "block",
        "continue:\nнет новых сигналов\n(pending или filled_now)",
        style="filled",
        fillcolor="#ffcdd2",
    )
    c.node("next", "следующий тик", style="filled", fillcolor="#d6e4f0")

dot.edge("start", "pend_fill")
dot.edge("pend_fill", "exec_now", label="да")
dot.edge("pend_fill", "pend_wait", label="нет")
dot.edge("pend_wait", "block", label="да\n(ещё не fill)")
dot.edge("block", "next")

# --- Пороги: один elif-блок (взаимоисключение) ---
with dot.subgraph(name="cluster_thresh") as c:
    c.attr(label="пороги (elif, одна ветка на тик)", style="rounded", color="#6b8e23", fontsize="11")
    c.node(
        "thresh",
        "порядок elif (взаимоисключение):\n"
        "1. long open: spread_long > thresh_open_long ∧ pos < target\n"
        "2. short open: spread_short > thresh_open_short ∧ pos < target\n"
        "3. long close: spread_short > thresh_close_long ∧ pos>0 ∧ whatpos=long\n"
        "4. short close: spread_long > thresh_close_short ∧ pos>0 ∧ whatpos=short",
        style="filled",
        fillcolor="#e8f5e9",
        fontsize="9",
        shape="box",
    )
    c.node("cand", "кандидат сигнала\n(+ n_signals_raw)", style="filled", fillcolor="#c5e1a5")

dot.edge("pend_wait", "thresh", label="нет pending")
dot.edge("thresh", "cand", label="сработала\nровно одна ветка")
dot.edge("thresh", "next", label="ничего\nне сработало")

# --- Гейты кандидата ---
with dot.subgraph(name="cluster_gates") as c:
    c.attr(label="гейты кандидата (отказ → ветка съедена)", style="rounded", color="#c45c26", fontsize="11")
    c.node(
        "gateA",
        "Gate A\nсвежесть? · потолки\nзадержки okx/bybit",
        style="filled",
        fillcolor="#ffe0b2",
    )
    c.node(
        "gateB",
        "Gate B\nокно avg + фильтр\nlatency точек;\nMA ≥ frac × порог",
        style="filled",
        fillcolor="#ffe0b2",
    )
    c.node(
        "vol",
        "объём\n(только если\nCheck_volume)",
        style="filled",
        fillcolor="#ffe0b2",
    )
    c.node(
        "eaten",
        "отказ гейта:\nветка съедена\n(без fall-through\nна другой open/close)",
        style="filled",
        fillcolor="#ffcdd2",
        fontsize="9",
    )

dot.edge("cand", "gateA")
dot.edge("gateA", "eaten", label="отказ")
dot.edge("gateA", "gateB", label="ok")
dot.edge("gateB", "eaten", label="отказ")
dot.edge("gateB", "vol", label="ok")
dot.edge("vol", "eaten", label="отказ")
dot.edge("eaten", "next")

# --- schedule / fill ---
with dot.subgraph(name="cluster_sched") as c:
    c.attr(label="schedule / fill (`Trade_Lat`)", style="rounded", color="#5c6bc0", fontsize="11")
    c.node(
        "sched",
        "найти fill_i:\nпервый тик с\nts ≥ signal + Trade_Lat",
        style="filled",
        fillcolor="#c5cae9",
    )
    c.node("miss", "нет fill_i →\nn_pending_missed", style="filled", fillcolor="#ffcdd2", fontsize="9")
    c.node("same", "fill_i == signal_i ?", shape="diamond", style="filled", fillcolor="#ffe6a7")
    c.node("imm", "исполнить сразу\nна этом тике", style="filled", fillcolor="#c8e6c9")
    c.node("pend", "pending = payload\n(ждать fill_i)", style="filled", fillcolor="#bbdefb")

dot.edge("vol", "sched", label="ok")
dot.edge("sched", "miss", label="не найден")
dot.edge("miss", "next")
dot.edge("sched", "same", label="есть fill_i")
dot.edge("same", "imm", label="да")
dot.edge("same", "pend", label="нет")
dot.edge("pend", "next")

# --- Эффект fill: два call-site ---
with dot.subgraph(name="cluster_fx") as c:
    c.attr(label="эффект fill (metric только на close)", style="rounded", color="#2e7d32", fontsize="11")
    c.node(
        "fx_def",
        "open → позиция / whatpos\nclose → Trade + metric += pnl",
        style="filled",
        fillcolor="#a5d6a7",
        fontsize="9",
    )
    c.node(
        "fx_imm",
        "open → позиция / whatpos\nclose → Trade + metric += pnl",
        style="filled",
        fillcolor="#a5d6a7",
        fontsize="9",
    )

dot.edge("exec_now", "fx_def", label="отложенный fill")
dot.edge("fx_def", "block")
dot.edge("imm", "fx_imm", label="немедленный fill")
dot.edge("fx_imm", "next")

# --- После цикла ---
with dot.subgraph(name="cluster_eod") as c:
    c.attr(label="после цикла", style="rounded", color="#616161", fontsize="11")
    c.node(
        "eod_pend",
        "pending остался →\nn_pending_missed",
        style="filled",
        fillcolor="#eeeeee",
        fontsize="9",
    )
    c.node(
        "eod_open",
        "open на конце ряда →\nв trades как status=open\nНЕ в metric",
        style="filled",
        fillcolor="#eeeeee",
        fontsize="9",
    )
    c.node("done", "BacktestResult", style="filled", fillcolor="#bdbdbd")

dot.edge("next", "start", label="ещё строки", style="dotted")
dot.edge("next", "eod_pend", label="конец ряда", style="bold")
dot.edge("eod_pend", "eod_open")
dot.edge("eod_open", "done")
dot.edge("cfg", "start", style="dashed", color="#888888", label="параметры")

dot


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Literal, Optional
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Единый снимок прогона; повторный Run сбрасывает «текучие» глобали.

base_coin = "LA"
start_date = "2026-07-21"
end_date = "2026-07-21"
max_points = 4000  # лимит точек линий графика
latency_hist_bins = 60

PARQUET_ROOT = Path(
    "/Users/mishatrubik/Desktop/spread/output/spreads_parquet_by_coins"
)
COIN_ROOT = PARQUET_ROOT / f"base_coin={base_coin}"


def _date_range_inclusive(start: str, end: str) -> list[str]:
    s = date.fromisoformat(start)
    e = date.fromisoformat(end)
    if e < s:
        raise ValueError(f"end_date {end} < start_date {start}")
    out: list[str] = []
    d = s
    while d <= e:
        out.append(d.isoformat())
        d += timedelta(days=1)
    return out


_wanted = _date_range_inclusive(start_date, end_date)
_frames: list[pd.DataFrame] = []
_dates_found: list[str] = []
_dates_missing: list[str] = []
for _d in _wanted:
    _p = COIN_ROOT / f"event_date={_d}"
    if _p.is_dir() and any(_p.glob("*.parquet")):
        _frames.append(pd.read_parquet(_p))
        _dates_found.append(_d)
    else:
        _dates_missing.append(_d)

if _dates_missing:
    warnings.warn(
        f"missing event_date partitions for {base_coin}: "
        f"missing={_dates_missing}; found={_dates_found}",
        stacklevel=1,
    )
if not _frames:
    raise ValueError(
        f"no parquet for {base_coin} in [{start_date}, {end_date}] under {COIN_ROOT}"
    )

df = pd.concat(_frames, ignore_index=True)
_sort_cols = [c for c in ("event_local_ts_ms", "event_dt") if c in df.columns]
if _sort_cols:
    df = df.sort_values(_sort_cols).reset_index(drop=True)

DATA = {
    "base_coin": base_coin,
    "start_date": start_date,
    "end_date": end_date,
    "dates_found": list(_dates_found),
    "dates_missing": list(_dates_missing),
    "max_points": max_points,
    "latency_hist_bins": latency_hist_bins,
    "COIN_ROOT": COIN_ROOT,
    "df": df,
}

VARIATION = {
    "thresh_open_long": 0.5,
    "thresh_open_short": 0.5,
    "thresh_close_long": 0.5,
    "thresh_close_short": 0.5,
    "open_frac": 0.7,   # mean_spread >= open_frac * thresh_open_*
    "close_frac": 0.7,  # mean_spread >= close_frac * thresh_close_*
}

# HYPER: latency = p95(объединённый df); freshness выкл.; fee — среднее за ногу.
if len(df) == 0:
    raise ValueError(
        f"empty df for {base_coin} [{start_date}, {end_date}]; cannot compute latency p95"
    )
for _col in ("okx_latency_ms", "bybit_latency_ms"):
    if _col not in df.columns:
        raise KeyError(f"missing {_col} in df columns={df.columns.tolist()}")
_p95_okx = float(df["okx_latency_ms"].quantile(0.95))
_p95_bybit = float(df["bybit_latency_ms"].quantile(0.95))
# NaN p95 → x > nan всегда False → гейт задержки молчит.
if not np.isfinite(_p95_okx) or not np.isfinite(_p95_bybit):
    raise ValueError(
        f"non-finite latency p95: okx={_p95_okx!r} bybit={_p95_bybit!r} "
        f"(okx_nan={int(df['okx_latency_ms'].isna().sum())}/{len(df)}, "
        f"bybit_nan={int(df['bybit_latency_ms'].isna().sum())}/{len(df)})"
    )

HYPER = {
    "max_freshness_ms": None,           # окно на будущее; не использовать
    "max_latency_okx_ms": _p95_okx,     # p95 okx_latency_ms
    "max_latency_bybit_ms": _p95_bybit, # p95 bybit_latency_ms
    "avg_window_sec": 2.0,              # ≫ характерного прострела; None = без гейта B
    "Trade_Lat": 100,                   # мс; временная константа до скрипта скорости ордера
    "Check_volume": False,              # True → гейт по position_size
    "position_size": 10.0,              # порог Bybit; OKX = /10; только при Check_volume
    "position_frac": 1.0,               # до гира 2.5
    "fee_rate": 0.00075,                # (0.0005+0.001)/2; ×4 ноги ≈ 0.3% одной биржи
}

thresh_open_long = VARIATION["thresh_open_long"]
thresh_open_short = VARIATION["thresh_open_short"]
thresh_close_long = VARIATION["thresh_close_long"]
thresh_close_short = VARIATION["thresh_close_short"]
open_frac = VARIATION["open_frac"]
close_frac = VARIATION["close_frac"]
max_freshness_ms = HYPER["max_freshness_ms"]
max_latency_okx_ms = HYPER["max_latency_okx_ms"]
max_latency_bybit_ms = HYPER["max_latency_bybit_ms"]
avg_window_sec = HYPER["avg_window_sec"]
Trade_Lat = HYPER["Trade_Lat"]
Check_volume = HYPER["Check_volume"]
position_size = HYPER["position_size"]
position_frac = HYPER["position_frac"]
fee_rate = HYPER["fee_rate"]

CONFIG = {"variation": VARIATION, "hyper": HYPER, "data": DATA}

_period = (
    start_date if start_date == end_date else f"{start_date}..{end_date}"
)
print(
    f"loaded {base_coin} {_period}: {len(df)} rows | "
    f"found={_dates_found} missing={_dates_missing} | root={COIN_ROOT}"
)
print(
    f"VARIATION open_frac={open_frac} close_frac={close_frac} | "
    f"HYPER p95_okx={max_latency_okx_ms:.3f} p95_bybit={max_latency_bybit_ms:.3f} "
    f"avg_window={avg_window_sec}s Trade_Lat={Trade_Lat}ms "
    f"fee_rate={fee_rate} position_frac={position_frac} Check_volume={Check_volume}"
)


In [ ]:
Side = Literal["long", "short"]
TradeStatus = Literal["closed", "open"]

LATENCY_WINDOW_RADIUS = 5  # тики ± вокруг сигнала для окна задержки

BOOK_SIZE_COLS = (
    "okx_bid_size",
    "okx_ask_size",
    "bybit_bid_size",
    "bybit_ask_size",
)


@dataclass
class Trade:
    """Одна сделка; `status='open'` на конце дня в `metric` не входит."""

    side: Side
    status: TradeStatus
    open_price: float
    open_ts: float
    open_dt: object
    quantity: float
    close_price: Optional[float] = None
    close_ts: Optional[float] = None
    close_dt: Optional[object] = None
    pnl: Optional[float] = None
    fees: float = 0.0
    okx_latency_ms_open: Optional[float] = None
    bybit_latency_ms_open: Optional[float] = None
    okx_latency_ms_close: Optional[float] = None
    bybit_latency_ms_close: Optional[float] = None
    okx_latency_ms_open_window: Optional[list[Optional[float]]] = None
    bybit_latency_ms_open_window: Optional[list[Optional[float]]] = None
    latency_window_offsets_open: Optional[list[int]] = None
    latency_window_truncated_open: bool = False
    okx_latency_ms_close_window: Optional[list[Optional[float]]] = None
    bybit_latency_ms_close_window: Optional[list[Optional[float]]] = None
    latency_window_offsets_close: Optional[list[int]] = None
    latency_window_truncated_close: bool = False
    signal_open_price: Optional[float] = None
    signal_open_ts: Optional[float] = None
    signal_open_dt: Optional[object] = None
    signal_close_price: Optional[float] = None
    signal_close_ts: Optional[float] = None
    signal_close_dt: Optional[object] = None
    open_fill_delay_ticks: int = 0
    close_fill_delay_ticks: int = 0


@dataclass
class BacktestResult:
    trades: list[Trade]
    metric: float
    open_position: Optional[Trade] = None
    n_signals_raw: int = 0
    n_signals_passed: int = 0
    n_filtered_by_freshness: int = 0
    n_filtered_by_latency: int = 0
    n_filtered_by_avg: int = 0
    n_filtered_by_size: int = 0
    n_pending_missed: int = 0  # отложенный fill за концом ряда
    fees_total: float = 0.0


def _latency_window_at(
    data: pd.DataFrame,
    i: int,
    *,
    radius: int = LATENCY_WINDOW_RADIUS,
) -> tuple[list[Optional[float]], list[Optional[float]], list[int], bool]:
    """Срез задержки okx/bybit вокруг индекса i: [i−radius, i+radius]."""
    n = len(data)
    lo = max(0, i - radius)
    hi = min(n, i + radius + 1)
    truncated = lo > (i - radius) or hi < (i + radius + 1)
    offsets = list(range(lo - i, hi - i))

    def _vals(col: str) -> list[Optional[float]]:
        out: list[Optional[float]] = []
        for x in data[col].iloc[lo:hi].tolist():
            out.append(None if pd.isna(x) else float(x))
        return out

    return _vals("okx_latency_ms"), _vals("bybit_latency_ms"), offsets, truncated


def _fee_cost_pct(quantity: float, fee_rate: float, *, legs: int = 2) -> float:
    """Комиссия в единицах spread-%: `fee_rate * 100 * legs * (quantity/100)`."""
    if fee_rate == 0.0 or quantity == 0.0:
        return 0.0
    return float(fee_rate) * 100.0 * float(legs) * (float(quantity) / 100.0)


def _book_cols_for(side: Side, *, is_open: bool) -> tuple[str, str]:
    """Вернуть `(okx_size_col, bybit_size_col)` для торгуемых сторон стакана."""
    if is_open:
        if side == "long":
            return "okx_ask_size", "bybit_bid_size"
        return "okx_bid_size", "bybit_ask_size"
    if side == "long":
        return "okx_bid_size", "bybit_ask_size"
    return "okx_ask_size", "bybit_bid_size"


def _require_size_columns(data: pd.DataFrame) -> None:
    """Жёсткая ошибка, если `Check_volume=True`, а колонок size нет."""
    missing = [c for c in BOOK_SIZE_COLS if c not in data.columns]
    if missing:
        raise ValueError(
            "Check_volume=True requires book size columns "
            f"{list(BOOK_SIZE_COLS)}, missing: {missing}. "
            "Set Check_volume=False until size data is available."
        )


def _volume_ok_bybit_ws(
    size_arrs: dict[str, np.ndarray],
    i: int,
    side: Side,
    *,
    is_open: bool,
    position_size: float,
) -> bool:
    """Гейт размера: Bybit > `position_size`, OKX > `position_size/10`; NaN → отказ."""
    c_okx, c_bybit = _book_cols_for(side, is_open=is_open)
    s_okx = size_arrs[c_okx][i]
    s_bybit = size_arrs[c_bybit][i]
    if s_okx != s_okx or s_bybit != s_bybit:
        return False
    return float(s_bybit) > float(position_size) and float(s_okx) > (
        float(position_size) / 10.0
    )


def _resolve_fill_index(
    signal_i: int,
    ts_ms: np.ndarray,
    *,
    Trade_Lat: float,
) -> Optional[int]:
    """Индекс причинного fill ≥ `signal_i`; `None`, если задержка не умещается в ряд."""
    n = len(ts_ms)
    delay = float(Trade_Lat)
    if delay <= 0:
        return signal_i
    target = float(ts_ms[signal_i]) + delay
    j = signal_i + 1
    while j < n and ts_ms[j] < target:
        j += 1
    return j if j < n else None


def _closed_trade(
    *,
    side: Side,
    open_price: float,
    open_ts: float,
    open_dt: object,
    close_price: float,
    close_ts: float,
    close_dt: object,
    quantity: float,
    fees: float,
    okx_latency_ms_open: Optional[float],
    bybit_latency_ms_open: Optional[float],
    okx_latency_ms_close: Optional[float],
    bybit_latency_ms_close: Optional[float],
    okx_latency_ms_open_window: Optional[list[Optional[float]]],
    bybit_latency_ms_open_window: Optional[list[Optional[float]]],
    latency_window_offsets_open: Optional[list[int]],
    latency_window_truncated_open: bool,
    okx_latency_ms_close_window: Optional[list[Optional[float]]],
    bybit_latency_ms_close_window: Optional[list[Optional[float]]],
    latency_window_offsets_close: Optional[list[int]],
    latency_window_truncated_close: bool,
    signal_open_price: Optional[float] = None,
    signal_open_ts: Optional[float] = None,
    signal_open_dt: Optional[object] = None,
    signal_close_price: Optional[float] = None,
    signal_close_ts: Optional[float] = None,
    signal_close_dt: Optional[object] = None,
    open_fill_delay_ticks: int = 0,
    close_fill_delay_ticks: int = 0,
) -> Trade:
    """Собрать закрытую сделку: `(open+close)*(quantity/100) − fees`."""
    pnl = (open_price + close_price) * (quantity / 100.0) - fees
    return Trade(
        side=side,
        status="closed",
        open_price=open_price,
        open_ts=open_ts,
        open_dt=open_dt,
        quantity=quantity,
        close_price=close_price,
        close_ts=close_ts,
        close_dt=close_dt,
        pnl=pnl,
        fees=fees,
        okx_latency_ms_open=okx_latency_ms_open,
        bybit_latency_ms_open=bybit_latency_ms_open,
        okx_latency_ms_close=okx_latency_ms_close,
        bybit_latency_ms_close=bybit_latency_ms_close,
        okx_latency_ms_open_window=okx_latency_ms_open_window,
        bybit_latency_ms_open_window=bybit_latency_ms_open_window,
        latency_window_offsets_open=latency_window_offsets_open,
        latency_window_truncated_open=latency_window_truncated_open,
        okx_latency_ms_close_window=okx_latency_ms_close_window,
        bybit_latency_ms_close_window=bybit_latency_ms_close_window,
        latency_window_offsets_close=latency_window_offsets_close,
        latency_window_truncated_close=latency_window_truncated_close,
        signal_open_price=signal_open_price if signal_open_price is not None else open_price,
        signal_open_ts=signal_open_ts if signal_open_ts is not None else open_ts,
        signal_open_dt=signal_open_dt if signal_open_dt is not None else open_dt,
        signal_close_price=signal_close_price if signal_close_price is not None else close_price,
        signal_close_ts=signal_close_ts if signal_close_ts is not None else close_ts,
        signal_close_dt=signal_close_dt if signal_close_dt is not None else close_dt,
        open_fill_delay_ticks=open_fill_delay_ticks,
        close_fill_delay_ticks=close_fill_delay_ticks,
    )



def compute_gate_b_ma(
    data: pd.DataFrame,
    *,
    avg_window_sec: float,
    max_latency_okx_ms: Optional[float] = None,
    max_latency_bybit_ms: Optional[float] = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Скользящее среднее Gate B по event_local_ts_ms; в среднее — только latency ≤ caps.
    Возвращает (ma_long, ma_short); NaN если в окне нет валидных точек.
    `data` должен быть отсортирован по времени.
    """
    if avg_window_sec <= 0:
        raise ValueError("avg_window_sec must be > 0")
    for col in (
        "event_local_ts_ms",
        "spread_long",
        "spread_short",
        "okx_latency_ms",
        "bybit_latency_ms",
    ):
        if col not in data.columns:
            raise KeyError(f"compute_gate_b_ma requires column {col!r}")
    n = len(data)
    ts_ms = data["event_local_ts_ms"].to_numpy(dtype="float64", copy=False)
    spread_long_arr = data["spread_long"].to_numpy(dtype="float64", copy=False)
    spread_short_arr = data["spread_short"].to_numpy(dtype="float64", copy=False)
    okx_lat_arr = data["okx_latency_ms"].to_numpy(dtype="float64", copy=False)
    bybit_lat_arr = data["bybit_latency_ms"].to_numpy(dtype="float64", copy=False)
    avg_valid = np.ones(n, dtype=bool)
    if max_latency_okx_ms is not None:
        avg_valid &= okx_lat_arr <= max_latency_okx_ms
    if max_latency_bybit_ms is not None:
        avg_valid &= bybit_lat_arr <= max_latency_bybit_ms
    window_ms = float(avg_window_sec) * 1000.0
    ma_long = np.full(n, np.nan, dtype="float64")
    ma_short = np.full(n, np.nan, dtype="float64")
    left = 0
    sum_long = 0.0
    sum_short = 0.0
    win_count = 0
    for i in range(n):
        t_lo = ts_ms[i] - window_ms
        while left <= i and ts_ms[left] < t_lo:
            if avg_valid[left]:
                sum_long -= spread_long_arr[left]
                sum_short -= spread_short_arr[left]
                win_count -= 1
            left += 1
        if avg_valid[i]:
            sum_long += spread_long_arr[i]
            sum_short += spread_short_arr[i]
            win_count += 1
        if win_count > 0:
            ma_long[i] = sum_long / win_count
            ma_short[i] = sum_short / win_count
    return ma_long, ma_short


def run_backtest(
    df: pd.DataFrame,
    thresh_open_long: float,
    thresh_open_short: float,
    thresh_close_long: float,
    thresh_close_short: float,
    *,
    open_frac: float = 1.0,
    close_frac: float = 1.0,
    max_freshness_ms: Optional[float] = None,
    max_latency_okx_ms: Optional[float] = None,
    max_latency_bybit_ms: Optional[float] = None,
    avg_window_sec: Optional[float] = None,
    Trade_Lat: float = 0.0,
    Check_volume: bool = False,
    position_size: float = 10.0,
    position_frac: float = 1.0,
    fee_rate: float = 0.0,
) -> BacktestResult:
    """Исторический цикл: `metric = sum(open+close)` по закрытым, минус комиссии."""
    if avg_window_sec is not None and avg_window_sec <= 0:
        raise ValueError("avg_window_sec must be > 0 when set")
    if not (0.0 < open_frac <= 1.0):
        raise ValueError("open_frac must be in (0, 1]")
    if not (0.0 < close_frac <= 1.0):
        raise ValueError("close_frac must be in (0, 1]")
    if not (0.0 < position_frac <= 1.0):
        raise ValueError("position_frac must be in (0, 1]")
    if fee_rate < 0.0:
        raise ValueError("fee_rate must be >= 0")
    if float(Trade_Lat) < 0.0:
        raise ValueError("Trade_Lat must be >= 0")
    if float(position_size) <= 0.0:
        raise ValueError("position_size must be > 0")
    if Check_volume:
        _require_size_columns(df)

    target_qty = 100.0 * float(position_frac)
    data = df.sort_values("event_dt").reset_index(drop=True)

    if max_freshness_ms is not None:
        for col in ("okx_freshness_ms", "bybit_freshness_ms"):
            if col not in data.columns:
                raise KeyError(f"Gate A freshness requires column {col!r}")
    need_latency_cols = (
        max_latency_okx_ms is not None
        or max_latency_bybit_ms is not None
        or avg_window_sec is not None
    )
    if need_latency_cols:
        for col in ("okx_latency_ms", "bybit_latency_ms"):
            if col not in data.columns:
                raise KeyError(f"latency/avg gates require column {col!r}")

    ts_ms = data["event_local_ts_ms"].to_numpy(dtype="float64", copy=False)
    spread_long_arr = data["spread_long"].to_numpy(dtype="float64", copy=False)
    spread_short_arr = data["spread_short"].to_numpy(dtype="float64", copy=False)
    okx_lat_arr = data["okx_latency_ms"].to_numpy(dtype="float64", copy=False)
    bybit_lat_arr = data["bybit_latency_ms"].to_numpy(dtype="float64", copy=False)
    if max_freshness_ms is not None:
        okx_fresh_arr = data["okx_freshness_ms"].to_numpy(dtype="float64", copy=False)
        bybit_fresh_arr = data["bybit_freshness_ms"].to_numpy(dtype="float64", copy=False)
    else:
        okx_fresh_arr = bybit_fresh_arr = None

    if Check_volume:
        size_arrs = {
            col: data[col].to_numpy(dtype="float64", copy=False) for col in BOOK_SIZE_COLS
        }
    else:
        size_arrs = None

    # Gate B MA — та же семантика, что compute_gate_b_ma / график.
    if avg_window_sec is not None:
        ma_long_arr, ma_short_arr = compute_gate_b_ma(
            data,
            avg_window_sec=float(avg_window_sec),
            max_latency_okx_ms=max_latency_okx_ms,
            max_latency_bybit_ms=max_latency_bybit_ms,
        )
    else:
        ma_long_arr = ma_short_arr = None

    def _mean_in_window(side_spread: str) -> Optional[float]:
        assert ma_long_arr is not None and ma_short_arr is not None
        v = ma_long_arr[i] if side_spread == "long" else ma_short_arr[i]
        if v != v:
            return None
        return float(v)

    def _gate_a_ok(i: int) -> tuple[bool, Optional[str]]:
        """Гейт A на тике i → `(ok, fail_reason)`."""
        if max_freshness_ms is not None:
            assert okx_fresh_arr is not None and bybit_fresh_arr is not None
            of_ = okx_fresh_arr[i]
            bf_ = bybit_fresh_arr[i]
            if of_ != of_ or bf_ != bf_ or of_ > max_freshness_ms or bf_ > max_freshness_ms:
                return False, "freshness"
        if max_latency_okx_ms is not None:
            ol_ = okx_lat_arr[i]
            if ol_ != ol_ or ol_ > max_latency_okx_ms:
                return False, "latency"
        if max_latency_bybit_ms is not None:
            bl_ = bybit_lat_arr[i]
            if bl_ != bl_ or bl_ > max_latency_bybit_ms:
                return False, "latency"
        return True, None

    def _gate_b_ok(mean_val: Optional[float], frac: float, thresh: float) -> bool:
        if avg_window_sec is None:
            return True
        if mean_val is None:
            return False
        return mean_val >= frac * thresh

    def _gate_volume_ok(i: int, side: Side, *, is_open: bool) -> bool:
        if not Check_volume:
            return True
        assert size_arrs is not None
        return _volume_ok_bybit_ws(
            size_arrs, i, side, is_open=is_open, position_size=position_size
        )

    trades: list[Trade] = []
    metric = 0.0
    fees_total = 0.0
    pos = 0.0
    whatpos: Optional[Side] = None
    open_price: Optional[float] = None
    open_ts: Optional[float] = None
    open_dt = None
    open_okx_lat: Optional[float] = None
    open_bybit_lat: Optional[float] = None
    open_okx_win: Optional[list[Optional[float]]] = None
    open_bybit_win: Optional[list[Optional[float]]] = None
    open_win_offsets: Optional[list[int]] = None
    open_win_trunc: bool = False
    signal_open_price: Optional[float] = None
    signal_open_ts: Optional[float] = None
    signal_open_dt = None
    open_fill_delay_ticks: int = 0
    open_fees: float = 0.0

    pending: Optional[dict] = None

    n_signals_raw = 0
    n_signals_passed = 0
    n_filtered_by_freshness = 0
    n_filtered_by_latency = 0
    n_filtered_by_avg = 0
    n_filtered_by_size = 0
    n_pending_missed = 0

    def _schedule_or_fill(kind: str, side: Side, signal_i: int, row) -> None:
        """Гейты пройдены → исполнить сразу или поставить причинно отложенный fill."""
        nonlocal pending, n_pending_missed, n_signals_passed
        fill_i = _resolve_fill_index(
            signal_i,
            ts_ms,
            Trade_Lat=Trade_Lat,
        )
        if fill_i is None:
            n_pending_missed += 1
            return
        n_signals_passed += 1
        payload = {
            "kind": kind,
            "side": side,
            "signal_i": signal_i,
            "fill_i": fill_i,
            "signal_row_spread_long": float(row.spread_long),
            "signal_row_spread_short": float(row.spread_short),
            "signal_ts": float(row.event_local_ts_ms),
            "signal_dt": row.event_dt,
            "signal_okx_lat": float(row.okx_latency_ms),
            "signal_bybit_lat": float(row.bybit_latency_ms),
        }
        if fill_i == signal_i:
            _execute_fill(payload, signal_i, row)
        else:
            pending = payload

    def _execute_fill(payload: dict, fill_i: int, fill_row) -> None:
        nonlocal pos, whatpos, open_price, open_ts, open_dt
        nonlocal open_okx_lat, open_bybit_lat, open_okx_win, open_bybit_win
        nonlocal open_win_offsets, open_win_trunc
        nonlocal signal_open_price, signal_open_ts, signal_open_dt
        nonlocal open_fill_delay_ticks, open_fees
        nonlocal metric, fees_total

        kind = payload["kind"]
        side: Side = payload["side"]
        signal_i = int(payload["signal_i"])
        delay_ticks = fill_i - signal_i

        if kind == "open":
            if side == "long":
                fill_spread = float(fill_row.spread_long)
                sig_spread = float(payload["signal_row_spread_long"])
            else:
                fill_spread = float(fill_row.spread_short)
                sig_spread = float(payload["signal_row_spread_short"])
            pos = target_qty
            open_price = fill_spread
            open_ts = float(fill_row.event_local_ts_ms)
            open_dt = fill_row.event_dt
            open_okx_lat = float(payload["signal_okx_lat"])
            open_bybit_lat = float(payload["signal_bybit_lat"])
            open_okx_win, open_bybit_win, open_win_offsets, open_win_trunc = (
                _latency_window_at(data, signal_i)
            )
            signal_open_price = sig_spread
            signal_open_ts = float(payload["signal_ts"])
            signal_open_dt = payload["signal_dt"]
            open_fill_delay_ticks = delay_ticks
            open_fees = _fee_cost_pct(target_qty, fee_rate, legs=2)
            whatpos = side
            return

        assert open_price is not None and open_ts is not None and whatpos == side
        if side == "long":
            fill_spread = float(fill_row.spread_short)
            sig_spread = float(payload["signal_row_spread_short"])
        else:
            fill_spread = float(fill_row.spread_long)
            sig_spread = float(payload["signal_row_spread_long"])
        close_okx_win, close_bybit_win, close_win_offsets, close_win_trunc = (
            _latency_window_at(data, signal_i)
        )
        close_fees = _fee_cost_pct(pos, fee_rate, legs=2)
        fees = open_fees + close_fees
        trade = _closed_trade(
            side=side,
            open_price=open_price,
            open_ts=open_ts,
            open_dt=open_dt,
            close_price=fill_spread,
            close_ts=float(fill_row.event_local_ts_ms),
            close_dt=fill_row.event_dt,
            quantity=pos,
            fees=fees,
            okx_latency_ms_open=open_okx_lat,
            bybit_latency_ms_open=open_bybit_lat,
            okx_latency_ms_close=float(payload["signal_okx_lat"]),
            bybit_latency_ms_close=float(payload["signal_bybit_lat"]),
            okx_latency_ms_open_window=open_okx_win,
            bybit_latency_ms_open_window=open_bybit_win,
            latency_window_offsets_open=open_win_offsets,
            latency_window_truncated_open=open_win_trunc,
            okx_latency_ms_close_window=close_okx_win,
            bybit_latency_ms_close_window=close_bybit_win,
            latency_window_offsets_close=close_win_offsets,
            latency_window_truncated_close=close_win_trunc,
            signal_open_price=signal_open_price,
            signal_open_ts=signal_open_ts,
            signal_open_dt=signal_open_dt,
            signal_close_price=sig_spread,
            signal_close_ts=float(payload["signal_ts"]),
            signal_close_dt=payload["signal_dt"],
            open_fill_delay_ticks=open_fill_delay_ticks,
            close_fill_delay_ticks=delay_ticks,
        )
        trades.append(trade)
        metric += float(trade.pnl)
        fees_total += fees
        pos = 0.0
        whatpos = None
        open_price = None
        open_ts = None
        open_dt = None
        open_okx_lat = None
        open_bybit_lat = None
        open_okx_win = None
        open_bybit_win = None
        open_win_offsets = None
        open_win_trunc = False
        signal_open_price = None
        signal_open_ts = None
        signal_open_dt = None
        open_fill_delay_ticks = 0
        open_fees = 0.0

    for i, row in enumerate(data.itertuples(index=False)):
        # Сначала закрываем отложенный fill; на этом тике новые сигналы не берём.
        filled_now = False
        if pending is not None and i == int(pending["fill_i"]):
            _execute_fill(pending, i, row)
            pending = None
            filled_now = True

        if pending is not None or filled_now:
            continue

        if row.spread_long > thresh_open_long and pos < target_qty:
            n_signals_raw += 1
            ok_a, reason = _gate_a_ok(i)
            if not ok_a:
                if reason == "freshness":
                    n_filtered_by_freshness += 1
                else:
                    n_filtered_by_latency += 1
            else:
                mean_l = (
                    _mean_in_window("long") if avg_window_sec is not None else None
                )
                if not _gate_b_ok(mean_l, open_frac, thresh_open_long):
                    n_filtered_by_avg += 1
                elif not _gate_volume_ok(i, "long", is_open=True):
                    n_filtered_by_size += 1
                else:
                    _schedule_or_fill("open", "long", i, row)
        elif row.spread_short > thresh_open_short and pos < target_qty:
            n_signals_raw += 1
            ok_a, reason = _gate_a_ok(i)
            if not ok_a:
                if reason == "freshness":
                    n_filtered_by_freshness += 1
                else:
                    n_filtered_by_latency += 1
            else:
                mean_s = (
                    _mean_in_window("short") if avg_window_sec is not None else None
                )
                if not _gate_b_ok(mean_s, open_frac, thresh_open_short):
                    n_filtered_by_avg += 1
                elif not _gate_volume_ok(i, "short", is_open=True):
                    n_filtered_by_size += 1
                else:
                    _schedule_or_fill("open", "short", i, row)
        elif row.spread_short > thresh_close_long and pos > 0 and whatpos == "long":
            n_signals_raw += 1
            ok_a, reason = _gate_a_ok(i)
            if not ok_a:
                if reason == "freshness":
                    n_filtered_by_freshness += 1
                else:
                    n_filtered_by_latency += 1
            else:
                mean_s = (
                    _mean_in_window("short") if avg_window_sec is not None else None
                )
                if not _gate_b_ok(mean_s, close_frac, thresh_close_long):
                    n_filtered_by_avg += 1
                elif not _gate_volume_ok(i, "long", is_open=False):
                    n_filtered_by_size += 1
                else:
                    _schedule_or_fill("close", "long", i, row)
        elif row.spread_long > thresh_close_short and pos > 0 and whatpos == "short":
            n_signals_raw += 1
            ok_a, reason = _gate_a_ok(i)
            if not ok_a:
                if reason == "freshness":
                    n_filtered_by_freshness += 1
                else:
                    n_filtered_by_latency += 1
            else:
                mean_l = (
                    _mean_in_window("long") if avg_window_sec is not None else None
                )
                if not _gate_b_ok(mean_l, close_frac, thresh_close_short):
                    n_filtered_by_avg += 1
                elif not _gate_volume_ok(i, "short", is_open=False):
                    n_filtered_by_size += 1
                else:
                    _schedule_or_fill("close", "short", i, row)

    if pending is not None:
        n_pending_missed += 1
        pending = None

    open_position: Optional[Trade] = None
    if whatpos is not None and open_price is not None and open_ts is not None:
        open_position = Trade(
            side=whatpos,
            status="open",
            open_price=open_price,
            open_ts=open_ts,
            open_dt=open_dt,
            quantity=pos,
            fees=open_fees,
            okx_latency_ms_open=open_okx_lat,
            bybit_latency_ms_open=open_bybit_lat,
            okx_latency_ms_open_window=open_okx_win,
            bybit_latency_ms_open_window=open_bybit_win,
            latency_window_offsets_open=open_win_offsets,
            latency_window_truncated_open=open_win_trunc,
            signal_open_price=signal_open_price,
            signal_open_ts=signal_open_ts,
            signal_open_dt=signal_open_dt,
            open_fill_delay_ticks=open_fill_delay_ticks,
        )
        trades.append(open_position)

    return BacktestResult(
        trades=trades,
        metric=metric,
        open_position=open_position,
        n_signals_raw=n_signals_raw,
        n_signals_passed=n_signals_passed,
        n_filtered_by_freshness=n_filtered_by_freshness,
        n_filtered_by_latency=n_filtered_by_latency,
        n_filtered_by_avg=n_filtered_by_avg,
        n_filtered_by_size=n_filtered_by_size,
        n_pending_missed=n_pending_missed,
        fees_total=fees_total,
    )


def build_backtest_kwargs(
    config: dict,
    *,
    gates: bool = False,
    execute: bool = False,
) -> dict:
    """Собрать kwargs для `run_backtest` из `CONFIG` и флагов режима."""
    v = config["variation"]
    h = config["hyper"]
    kwargs: dict = {
        "thresh_open_long": v["thresh_open_long"],
        "thresh_open_short": v["thresh_open_short"],
        "thresh_close_long": v["thresh_close_long"],
        "thresh_close_short": v["thresh_close_short"],
    }
    if gates:
        kwargs.update(
            open_frac=v["open_frac"],
            close_frac=v["close_frac"],
            max_freshness_ms=h["max_freshness_ms"],
            max_latency_okx_ms=h["max_latency_okx_ms"],
            max_latency_bybit_ms=h["max_latency_bybit_ms"],
            avg_window_sec=h["avg_window_sec"],
        )
    if execute:
        kwargs.update(
            Trade_Lat=h["Trade_Lat"],
            Check_volume=h["Check_volume"],
            position_size=h["position_size"],
            position_frac=h["position_frac"],
            fee_rate=h["fee_rate"],
        )
    return kwargs


def run_backtest_from_config(
    config: dict,
    *,
    gates: bool = False,
    execute: bool = False,
) -> BacktestResult:
    """Прогон из `CONFIG`: один набор чисел, режимы — только флаги."""
    return run_backtest(config["data"]["df"], **build_backtest_kwargs(
        config, gates=gates, execute=execute
    ))


def trademodel(
    thresh_open_long: float,
    thresh_open_short: float,
    thresh_close_long: float,
    thresh_close_short: float,
) -> tuple[list[Trade], float]:
    """Прогон с порогами; остальное из `CONFIG` (режим `gear1`)."""
    if "CONFIG" not in globals():
        raise RuntimeError("CONFIG не найден — сначала выполните ячейку конфига")
    cfg = {
        "variation": {
            **CONFIG["variation"],
            "thresh_open_long": thresh_open_long,
            "thresh_open_short": thresh_open_short,
            "thresh_close_long": thresh_close_long,
            "thresh_close_short": thresh_close_short,
        },
        "hyper": CONFIG["hyper"],
        "data": CONFIG["data"],
    }
    result = run_backtest_from_config(cfg, gates=True, execute=True)
    return result.trades, result.metric


def run_backtest_from_params(
    df: pd.DataFrame,
    variation: dict,
    hyper: dict,
) -> BacktestResult:
    """Обёртка `variation`+`hyper` для будущего `objective(params)`."""
    return run_backtest(
        df,
        variation["thresh_open_long"],
        variation["thresh_open_short"],
        variation["thresh_close_long"],
        variation["thresh_close_short"],
        open_frac=variation["open_frac"],
        close_frac=variation["close_frac"],
        max_freshness_ms=hyper["max_freshness_ms"],
        max_latency_okx_ms=hyper["max_latency_okx_ms"],
        max_latency_bybit_ms=hyper["max_latency_bybit_ms"],
        avg_window_sec=hyper["avg_window_sec"],
        Trade_Lat=hyper["Trade_Lat"],
        Check_volume=hyper["Check_volume"],
        position_size=hyper["position_size"],
        position_frac=hyper["position_frac"],
        fee_rate=hyper["fee_rate"],
    )


In [ ]:
def _downsample_for_plot(
    data: pd.DataFrame,
    max_points: Optional[int] = None,
) -> pd.DataFrame:
    """Прореживание ряда спреда только для линий графика."""
    if max_points is None or max_points <= 0 or len(data) <= max_points:
        return data
    step = max(1, len(data) // max_points)
    idx = list(range(0, len(data), step))
    if idx[-1] != len(data) - 1:
        idx.append(len(data) - 1)
    return data.iloc[idx]


def plot_strategy(
    df: pd.DataFrame,
    trades: Optional[list[Trade]] = None,
    *,
    title: Optional[str] = None,
    width: int = 1400,
    height: int = 700,
    max_points: Optional[int] = None,
    marker_mode: str = "both",
    avg_window_sec: object = ...,
    max_latency_okx_ms: object = ...,
    max_latency_bybit_ms: object = ...,
) -> go.Figure:
    """График спредов + MA Gate B и маркеры входа/выхода."""
    if marker_mode not in ("fill", "signal", "both"):
        raise ValueError("marker_mode must be 'fill', 'signal', or 'both'")
    if max_points is None:
        max_points = int(CONFIG["data"].get("max_points", 4000)) if "CONFIG" in globals() else 4000
    if avg_window_sec is ...:
        avg_window_sec = CONFIG["hyper"].get("avg_window_sec") if "CONFIG" in globals() else None
    if max_latency_okx_ms is ...:
        max_latency_okx_ms = CONFIG["hyper"].get("max_latency_okx_ms") if "CONFIG" in globals() else None
    if max_latency_bybit_ms is ...:
        max_latency_bybit_ms = CONFIG["hyper"].get("max_latency_bybit_ms") if "CONFIG" in globals() else None
    if title is None:
        coin = CONFIG["data"].get("base_coin", "?") if "CONFIG" in globals() else "?"
        if "CONFIG" in globals():
            sd = CONFIG["data"].get("start_date", "?")
            ed = CONFIG["data"].get("end_date", sd)
            period = sd if sd == ed else f"{sd}..{ed}"
        else:
            period = "?"
        title = f"{coin} {period} — спреды и входы/выходы"

    data = df.sort_values("event_dt").reset_index(drop=True)
    if avg_window_sec is not None:
        ma_long, ma_short = compute_gate_b_ma(
            data,
            avg_window_sec=float(avg_window_sec),
            max_latency_okx_ms=max_latency_okx_ms,
            max_latency_bybit_ms=max_latency_bybit_ms,
        )
        data = data.assign(_ma_long=ma_long, _ma_short=ma_short)
    plot_data = _downsample_for_plot(data, max_points=max_points)
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=plot_data["event_dt"], y=plot_data["spread_long"],
        mode="lines", name="spread_long", line=dict(width=1.4, color="#1f77b4"),
    ))
    fig.add_trace(go.Scattergl(
        x=plot_data["event_dt"], y=plot_data["spread_short"],
        mode="lines", name="spread_short", line=dict(width=1.4, color="#d62728"),
    ))
    if avg_window_sec is not None:
        fig.add_trace(go.Scattergl(
            x=plot_data["event_dt"], y=plot_data["_ma_long"],
            mode="lines", name=f"MA long ({avg_window_sec}s)",
            line=dict(width=1.6, color="#17becf", dash="dash"),
        ))
        fig.add_trace(go.Scattergl(
            x=plot_data["event_dt"], y=plot_data["_ma_short"],
            mode="lines", name=f"MA short ({avg_window_sec}s)",
            line=dict(width=1.6, color="#bcbd22", dash="dash"),
        ))
    else:
        fig.add_annotation(
            text="Gate B MA: off (avg_window_sec=None)",
            xref="paper", yref="paper", x=0.01, y=0.99,
            showarrow=False, font=dict(size=11, color="#666"),
        )

    def _add_marker_trace(*, name, xs, ys, hover, symbol, color, size):
        if not xs:
            return
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode="markers", name=name,
            marker=dict(symbol=symbol, color=color, size=size, line=dict(width=1, color="#222")),
            text=hover, hovertemplate="%{text}<extra></extra>",
        ))

    if trades:
        side_colors = {"long": "#2ca02c", "short": "#ff7f0e"}
        close_colors = {"long": "#1f77b4", "short": "#d62728"}
        closed = [t for t in trades if t.status == "closed"]
        eod = [t for t in trades if t.status == "open"]
        show_fill = marker_mode in ("fill", "both")
        show_signal = marker_mode in ("signal", "both")

        for side in ("long", "short"):
            group = [t for t in closed if t.side == side]
            if not group:
                continue
            if show_fill:
                _add_marker_trace(
                    name=f"{side} open (fill)",
                    xs=[t.open_dt for t in group],
                    ys=[t.open_price for t in group],
                    hover=[
                        (
                            f"{t.side} open исполнение"
                            f"<br>fill={t.open_price:.6f} ts={t.open_ts}"
                            f"<br>signal={t.signal_open_price} ts={t.signal_open_ts}"
                            f"<br>delay_ticks={t.open_fill_delay_ticks}"
                        )
                        for t in group
                    ],
                    symbol="triangle-up", color=side_colors[side], size=12,
                )
                _add_marker_trace(
                    name=f"{side} close (fill)",
                    xs=[t.close_dt for t in group],
                    ys=[t.close_price for t in group],
                    hover=[
                        (
                            f"{t.side} close исполнение"
                            f"<br>fill={t.close_price:.6f} ts={t.close_ts}"
                            f"<br>signal={t.signal_close_price} ts={t.signal_close_ts}"
                            f"<br>pnl={t.pnl:.6f} delay_ticks={t.close_fill_delay_ticks}"
                        )
                        for t in group
                    ],
                    symbol="triangle-down", color=close_colors[side], size=12,
                )
            if show_signal:
                _add_marker_trace(
                    name=f"{side} open (signal)",
                    xs=[t.signal_open_dt if t.signal_open_dt is not None else t.open_dt for t in group],
                    ys=[t.signal_open_price if t.signal_open_price is not None else t.open_price for t in group],
                    hover=[
                        (
                            f"{t.side} open сигнал"
                            f"<br>signal={t.signal_open_price} ts={t.signal_open_ts}"
                            f"<br>fill={t.open_price:.6f} ts={t.open_ts}"
                            f"<br>delay_ticks={t.open_fill_delay_ticks}"
                        )
                        for t in group
                    ],
                    symbol="circle-open", color=side_colors[side], size=10,
                )
                _add_marker_trace(
                    name=f"{side} close (signal)",
                    xs=[t.signal_close_dt if t.signal_close_dt is not None else t.close_dt for t in group],
                    ys=[t.signal_close_price if t.signal_close_price is not None else t.close_price for t in group],
                    hover=[
                        (
                            f"{t.side} close сигнал"
                            f"<br>signal={t.signal_close_price} ts={t.signal_close_ts}"
                            f"<br>fill={t.close_price:.6f} ts={t.close_ts}"
                            f"<br>pnl={t.pnl:.6f}"
                        )
                        for t in group
                    ],
                    symbol="circle-open", color=close_colors[side], size=10,
                )

        for side in ("long", "short"):
            group = [t for t in eod if t.side == side]
            if not group:
                continue
            _add_marker_trace(
                name=f"{side} open (EOD fill)",
                xs=[t.open_dt for t in group],
                ys=[t.open_price for t in group],
                hover=[
                    (
                        f"{t.side} open конец-дня исполнение"
                        f"<br>fill={t.open_price:.6f} ts={t.open_ts}"
                        f"<br>signal={t.signal_open_price} ts={t.signal_open_ts}"
                    )
                    for t in group
                ],
                symbol="diamond", color=side_colors[side], size=13,
            )
            if show_signal:
                _add_marker_trace(
                    name=f"{side} open (EOD signal)",
                    xs=[t.signal_open_dt if t.signal_open_dt is not None else t.open_dt for t in group],
                    ys=[t.signal_open_price if t.signal_open_price is not None else t.open_price for t in group],
                    hover=[
                        (
                            f"{t.side} open конец-дня сигнал"
                            f"<br>signal={t.signal_open_price} ts={t.signal_open_ts}"
                            f"<br>fill={t.open_price:.6f}"
                        )
                        for t in group
                    ],
                    symbol="diamond-open", color=side_colors[side], size=11,
                )

    fig.update_layout(
        title=title, width=width, height=height,
        xaxis_title="event_dt", yaxis_title="spread",
        hovermode="closest", legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    return fig


# --- прогон: только из CONFIG ---
if "CONFIG" not in globals():
    raise RuntimeError("CONFIG не найден. Сначала выполните ячейку конфига.")

VARIATION = CONFIG["variation"]
HYPER = CONFIG["hyper"]
DATA = CONFIG["data"]
base_coin = DATA["base_coin"]
start_date = DATA["start_date"]
end_date = DATA["end_date"]
_period = start_date if start_date == end_date else f"{start_date}..{end_date}"
max_points = DATA["max_points"]
df = DATA["df"]
Trade_Lat = HYPER["Trade_Lat"]
Check_volume = HYPER["Check_volume"]
position_frac = HYPER["position_frac"]
fee_rate = HYPER["fee_rate"]
open_frac = VARIATION["open_frac"]
close_frac = VARIATION["close_frac"]
max_freshness_ms = HYPER["max_freshness_ms"]
max_latency_okx_ms = HYPER["max_latency_okx_ms"]
max_latency_bybit_ms = HYPER["max_latency_bybit_ms"]
avg_window_sec = HYPER["avg_window_sec"]

result_off = run_backtest_from_config(CONFIG, gates=False, execute=False)
result_gear1 = run_backtest_from_config(CONFIG, gates=True, execute=True)
result_off_trade_lat = run_backtest_from_config(CONFIG, gates=False, execute=True)


def _summarize(label: str, r: BacktestResult) -> None:
    closed_n = sum(1 for t in r.trades if t.status == "closed")
    open_n = 1 if r.open_position is not None else 0
    print(
        f"{label}: metric={r.metric:.6f}  closed={closed_n}  open_at_end={open_n}  "
        f"raw={r.n_signals_raw}  passed={r.n_signals_passed}  "
        f"filt_fresh={r.n_filtered_by_freshness}  filt_lat={r.n_filtered_by_latency}  "
        f"filt_avg={r.n_filtered_by_avg}  filt_size={r.n_filtered_by_size}  "
        f"pending_miss={r.n_pending_missed}  fees={r.fees_total:.6f}"
    )


def _n_closed(r: BacktestResult) -> int:
    return sum(1 for t in r.trades if t.status == "closed")


def _exec_keys(r_kwargs: dict) -> dict:
    return {
        k: r_kwargs[k]
        for k in ("Trade_Lat", "Check_volume", "position_size", "position_frac", "fee_rate")
        if k in r_kwargs
    }


_kw_g1 = build_backtest_kwargs(CONFIG, gates=True, execute=True)
_kw_off_tl = build_backtest_kwargs(CONFIG, gates=False, execute=True)
assert _exec_keys(_kw_g1) == _exec_keys(_kw_off_tl), (_exec_keys(_kw_g1), _exec_keys(_kw_off_tl))

print(
    f"gear1 gates: freshness<={max_freshness_ms}  "
    f"lat_okx<={max_latency_okx_ms}  lat_bybit<={max_latency_bybit_ms}  "
    f"avg_window={avg_window_sec}s  open_frac={open_frac}  close_frac={close_frac}"
)
print(
    f"execute (gear1 / OFF+Trade_Lat): Trade_Lat={Trade_Lat}  "
    f"Check_volume={Check_volume}  position_frac={position_frac}  fee_rate={fee_rate}"
)
_summarize("OFF           ", result_off)
_summarize("gear1         ", result_gear1)
_summarize("OFF+Trade_Lat ", result_off_trade_lat)

_closed_off = _n_closed(result_off)
_closed_off_tl = _n_closed(result_off_trade_lat)
assert _closed_off_tl <= _closed_off, (
    f"OFF+Trade_Lat closed={_closed_off_tl} > OFF closed={_closed_off} — баг fill"
)
print(f"ok: closed OFF={_closed_off} ≥ OFF+Trade_Lat={_closed_off_tl} (инвариант ≤ OFF)")

_closed = [t for t in result_gear1.trades if t.status == "closed"]
_eod = [t for t in result_gear1.trades if t.status == "open"]
assert all(t.signal_open_dt is not None for t in result_gear1.trades)
assert all(t.signal_close_dt is not None for t in _closed)
print(
    f"проверка маркеров: closed={len(_closed)} → пары open+close; "
    f"eod_open={len(_eod)}; signal_dt заполнены"
)

result = result_gear1
plot_strategy(
    df,
    result.trades,
    title=(
        f"{base_coin} {_period} — gear1 + Trade_Lat={Trade_Lat}ms "
        f"Check_volume={Check_volume}"
    ),
    max_points=max_points,
    marker_mode="both",
    avg_window_sec=avg_window_sec,
    max_latency_okx_ms=max_latency_okx_ms,
    max_latency_bybit_ms=max_latency_bybit_ms,
)


## Приложение: задержки доставки (не ядро гира 1)

Опционально — гистограмма `okx_latency_ms` / `bybit_latency_ms` для проверки p95 в `HYPER`.


In [ ]:
def plot_latency_hist(
    df: pd.DataFrame,
    *,
    title: Optional[str] = None,
    width: int = 1000,
    height: int = 450,
    nbins: Optional[int] = None,
) -> go.Figure:
    """Наложенные гистограммы задержки доставки OKX/Bybit за период."""
    if nbins is None:
        nbins = int(CONFIG["data"].get("latency_hist_bins", 60)) if "CONFIG" in globals() else 60
    if title is None:
        coin = CONFIG["data"].get("base_coin", "?") if "CONFIG" in globals() else "?"
        if "CONFIG" in globals():
            sd = CONFIG["data"].get("start_date", "?")
            ed = CONFIG["data"].get("end_date", sd)
            period = sd if sd == ed else f"{sd}..{ed}"
        else:
            period = "?"
        title = f"{coin} {period} — гистограмма задержки (мс)"

    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=df["okx_latency_ms"], name="okx_latency_ms", nbinsx=nbins,
        opacity=0.55, marker_color="#9467bd",
    ))
    fig.add_trace(go.Histogram(
        x=df["bybit_latency_ms"], name="bybit_latency_ms", nbinsx=nbins,
        opacity=0.55, marker_color="#8c564b",
    ))
    fig.update_layout(
        title=title, width=width, height=height, barmode="overlay",
        xaxis_title="latency_ms", yaxis_title="count",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    return fig


plot_latency_hist(
    df,
    title=f"{base_coin} {_period} — гистограмма задержки",
    nbins=DATA.get("latency_hist_bins", latency_hist_bins),
)


## Валидация сделок (`result_gear1`)

По каждой сделке из прогона `gear1` (`result` / `result_gear1`): текстовая сводка и окна **±5 с** вокруг **исполнения** (`fill`). На графике открытия/закрытия — спред, вызвавший сигнал, и скользящее среднее `Gate B` по тому же спреду; все тики окна без прореживания. Вертикали: сигнал и `fill` (если различаются). Для `open` на конце ряда — только окно открытия.

In [ ]:
# Валидация сделок gear1: сводка + окна ±5с вокруг fill (без downsample)
_VALIDATION_HALF_MS = 5_000

_bt = result_gear1 if "result_gear1" in globals() else result
_trades_v = list(_bt.trades)
if not _trades_v:
    print("нет сделок для валидации")
else:
    _df_v = df.sort_values("event_local_ts_ms").reset_index(drop=True)
    _ts_v = _df_v["event_local_ts_ms"].to_numpy(dtype="float64", copy=False)
    _h = CONFIG["hyper"]
    _avg_w = _h.get("avg_window_sec")
    if _avg_w is not None:
        _ma_l, _ma_s = compute_gate_b_ma(
            _df_v,
            avg_window_sec=float(_avg_w),
            max_latency_okx_ms=_h.get("max_latency_okx_ms"),
            max_latency_bybit_ms=_h.get("max_latency_bybit_ms"),
        )
        _df_v = _df_v.assign(_ma_long=_ma_l, _ma_short=_ma_s)

    def _spread_cols(side: str, *, is_open: bool) -> tuple[str, str]:
        # open long / close short → spread_long; open short / close long → spread_short
        use_long = (side == "long") if is_open else (side == "short")
        return ("spread_long", "_ma_long") if use_long else ("spread_short", "_ma_short")

    def _window(center_ms: float) -> pd.DataFrame:
        lo, hi = center_ms - _VALIDATION_HALF_MS, center_ms + _VALIDATION_HALF_MS
        return _df_v[(_ts_v >= lo) & (_ts_v <= hi)]

    def _fmt_dt(x) -> str:
        return "—" if x is None else str(x)

    def _plot_event(
        trade: Trade,
        *,
        is_open: bool,
        trade_i: int,
    ) -> None:
        if is_open:
            fill_ts, sig_ts = float(trade.open_ts), trade.signal_open_ts
            fill_dt, sig_dt = trade.open_dt, trade.signal_open_dt
            fill_px, sig_px = trade.open_price, trade.signal_open_price
            delay = trade.open_fill_delay_ticks
            kind = "open"
        else:
            if trade.close_ts is None:
                return
            fill_ts, sig_ts = float(trade.close_ts), trade.signal_close_ts
            fill_dt, sig_dt = trade.close_dt, trade.signal_close_dt
            fill_px, sig_px = trade.close_price, trade.signal_close_price
            delay = trade.close_fill_delay_ticks
            kind = "close"

        spr_col, ma_col = _spread_cols(trade.side, is_open=is_open)
        w = _window(fill_ts)  # центр = fill
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=w["event_dt"], y=w[spr_col],
            mode="lines+markers", name=spr_col,
            line=dict(width=1.5), marker=dict(size=4),
        ))
        if _avg_w is not None and ma_col in w.columns:
            fig.add_trace(go.Scatter(
                x=w["event_dt"], y=w[ma_col],
                mode="lines", name=f"MA Gate B ({_avg_w}s)",
                line=dict(width=1.6, dash="dash"),
            ))
        # вертикали: fill (центр) и signal, если отличается
        fig.add_vline(
            x=fill_dt, line_width=2, line_dash="solid", line_color="#222",
            annotation_text="fill", annotation_position="top left",
        )
        if sig_ts is not None and abs(float(sig_ts) - fill_ts) > 1e-6:
            fig.add_vline(
                x=sig_dt, line_width=1.5, line_dash="dot", line_color="#e377c2",
                annotation_text="signal", annotation_position="top right",
            )
        fig.update_layout(
            title=(
                f"#{trade_i} {trade.side} {kind} | fill={_fmt_dt(fill_dt)} "
                f"px={fill_px:.4f} | signal={_fmt_dt(sig_dt)} "
                f"px={sig_px if sig_px is None else f'{sig_px:.4f}'} | "
                f"delay_ticks={delay} | ticks={len(w)}"
            ),
            width=1000, height=360,
            margin=dict(l=50, r=20, t=60, b=40),
            legend=dict(orientation="h", y=1.12),
        )
        fig.show()

    print(f"валидация: {len(_trades_v)} сделок из result_gear1; окно ±5с, центр=fill")
    for _i, _t in enumerate(_trades_v, start=1):
        _hold = None
        if _t.close_ts is not None:
            _hold = (float(_t.close_ts) - float(_t.open_ts)) / 1000.0
        print("=" * 72)
        print(
            f"#{_i} side={_t.side} status={_t.status} qty={_t.quantity}\n"
            f"  signal_open={_fmt_dt(_t.signal_open_dt)}  fill_open={_fmt_dt(_t.open_dt)}  "
            f"open_px={_t.open_price:.4f}  sig_open_px={_t.signal_open_price}\n"
            f"  signal_close={_fmt_dt(_t.signal_close_dt)}  fill_close={_fmt_dt(_t.close_dt)}  "
            f"close_px={_t.close_price}  sig_close_px={_t.signal_close_price}\n"
            f"  pnl={_t.pnl}  fees={_t.fees:.6f}  "
            f"open_delay_ticks={_t.open_fill_delay_ticks}  "
            f"close_delay_ticks={_t.close_fill_delay_ticks}\n"
            f"  lat_open okx/bybit={_t.okx_latency_ms_open}/{_t.bybit_latency_ms_open}  "
            f"lat_close okx/bybit={_t.okx_latency_ms_close}/{_t.bybit_latency_ms_close}\n"
            f"  hold_sec={_hold if _hold is None else f'{_hold:.3f}'}"
        )
        _plot_event(_t, is_open=True, trade_i=_i)
        if _t.status == "closed":
            _plot_event(_t, is_open=False, trade_i=_i)
        else:
            print(f"  #{_i}: open на конце ряда — график закрытия пропущен")
